<a href="https://colab.research.google.com/github/archipelagoing/Situationion/blob/main/v6TightClean_RelationalDynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SituatiONION V6
## Relational Migration, Abstraction, Geometry, and Interpretability

### Central research question

> **How do control-validated situation relations develop across GPT-2 XL depth, and how are relational abstraction, representational migration, and geometric transformation associated?**

### Core variables

- **Relational:** `role_assignment`, `cause_holder_role`, `temporal_relation`
- **Simple comparisons:** `event_state`, `polarity`
- **Readouts:** `changed_token`, `event_token`, `final_token`, `agent_recipient`, `mean_pool`, `max_pool`
- **Model:** GPT-2 XL, 48 transformer layers

### Primary analysis map

| Section | Question | Primary metric | Primary null / comparison | Uncertainty | Required figure |
|---|---|---|---|---|---|
| V6.1 Migration | Does control-adjusted accessibility redistribute across readouts with depth? | $M_R(l)=JS(P_{R,l},P_{R,l+1})$ | relational vs. simple tasks | example-level bootstrap where valid | migration-by-layer + accessibility heatmap |
| V6.2 Abstraction | Does relational information survive systematic novelty? | $G_R(l,k)$ = conservative advantage under holdout $k$ | matched V5 controls + harder holdouts | bootstrap over test examples | layer × holdout abstraction map |
| V6.3 Geometry | Does relational geometry transform across depth? | $D_R(l)=d_G(S_R(l),S_R(l+1))$ | simple-variable geometry + within-relation stability | bootstrap / permutation where valid | adjacent geometric movement + cross-layer matrix |
| V6.4 Coupling | Are migration and geometric transformation associated? | association between $M_R(l)$ and $D_R(l)$ | shuffled transition pairing / task-aware null | bootstrap CI | $M$ vs. $D$ transition scatter |
| Relation Lens | Can hidden states map into a stable interpretable relational space? | held-out prototype/lens recovery | entity-held-out performance | entity/example bootstrap | lens recovery by layer |
| Integration | Where does abstraction emerge relative to migration and geometry? | preregistered $M\leftrightarrow G$, $D\leftrightarrow G$ | task-aware / layer-shuffled nulls | bootstrap CI | integrated trajectory panel |

### Operational definition of migration

$$
\boxed{
M_R(l)=JS\!\left(P_{R,l},P_{R,l+1}\right)
}
$$

where $P_{R,l}$ is the normalized distribution of **positive conservative advantage across readouts** for relation $R$ at layer $l$.

**Migration is descriptive, not causal.** It means redistribution of control-adjusted accessibility across readouts, not literal information transport.

### Scope boundary

V6 is observational/representational. Activation patching, ablation, causal tracing, mediation, and circuit claims remain V7.

### Literature anchors

- Chanin, Hunter & Camburu (2024), *Identifying Linear Relational Concepts in Large Language Models*
- Petty et al. (2024), *The Impact of Depth on Compositional Generalization in Transformer Language Models*
- Wold et al. (2024), *Compositional Generalization with Grounded Language Models*
- Morand, Mothe & Piwowarski (2025), *On the Representations of Entities in Auto-regressive Large Language Models*
- Chang, Deng & Chen (2025), *The Generalization Ridge: Information Flow in Natural Language Generation*
- Sakata et al. (2026), *Linear Representations of Hierarchical Concepts in Language Models*

Full references appear at the end.

# PART I — V5 Foundation

**Goal:** reproduce the exact V5 quantities V6 depends on before introducing any new metric.

**Exit criterion:** V5 artifacts load, schema checks pass, conservative advantage is reconstructed, and hidden-state readouts reproduce the V5 representation definitions.

## 1. Imports and Configuration

Define the model, tasks, readouts, random seed, cache paths, output paths, and analysis constants here.

Keep all configuration centralized so the notebook is reproducible and easy to rerun.

In [ ]:
# Imports and global configuration — aligned to the actual V5 notebook

from pathlib import Path
import hashlib
import json
import math
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from scipy import sparse
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr
from scipy.linalg import subspace_angles

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

SEED = 7                     # Preserve V5 seed for direct V5↔V6 comparisons
MODEL_NAME = "gpt2-xl"
N_LAYERS = 48
HIDDEN_SIZE = 1600

RELATIONAL_TASKS = [
    "role_assignment",
    "cause_holder_role",
    "temporal_relation",
]

CONTROL_TASKS = [
    "event_state",
    "polarity",
]

ALL_TASKS = RELATIONAL_TASKS + CONTROL_TASKS

# Exact V5 task schema
TASKS = {
    "agent_identity": ("semantic_agent", "agent_identity_eligible"),
    "recipient_identity": ("semantic_recipient", "recipient_identity_eligible"),
    "role_assignment": ("role_assignment", "role_assignment_eligible"),
    "event_state": ("event_state", "event_state_eligible"),
    "cause_holder_role": ("cause_holder_role", "cause_holder_role_eligible"),
    "polarity": ("polarity", "polarity_eligible"),
    "temporal_relation": ("temporal_relation", "temporal_relation_eligible"),
}

READOUTS = (
    "changed_token",
    "event_token",
    "final_token",
    "agent_recipient",
    "mean_pool",
    "max_pool",
)

LAYERS = range(N_LAYERS)

ROOT = Path(".")
RESULTS_DIR = ROOT / "results_v6"
FIGURE_DIR = ROOT / "figures_v6"
CACHE_DIR = ROOT / "cache_v6"

for p in [RESULTS_DIR, FIGURE_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)

print("Configured V6 from the actual V5 schema.")
print("Relational tasks:", RELATIONAL_TASKS)
print("Simple controls:", CONTROL_TASKS)
print("Readouts:", READOUTS)


## 2. Load V5 Artifacts

Load the four V5 inputs that V6 needs:

- `v5_atlas`
- `v5_hidden_states`
- `v5_stimulus_metadata`
- `v5_split_metadata`

Then verify that the expected tasks, layers, readouts, and hidden dimensionality are present.

If your actual V5 filenames differ, only edit this cell.

In [ ]:
# Load the ACTUAL V5 artifacts.
#
# V5 used:
#   data/v5_probe_examples.csv
#   results/layer_curves/gpt2xl_hidden_states.pt
#   .../v5_decodability/decodability_atlas.csv
#
# This cell searches several likely locations so it works in a local clone
# or a Colab/Drive project directory without changing the analysis code.

def first_existing(candidates, label):
    candidates = [Path(p) for p in candidates]
    for p in candidates:
        if p.exists():
            print(f"✓ {label}: {p}")
            return p
    print(f"✗ Could not find {label}. Checked:")
    for p in candidates:
        print("   ", p)
    return None

DATA_PATH = first_existing(
    [
        ROOT / "data" / "v5_probe_examples.csv",
        ROOT / "v5" / "data" / "v5_probe_examples.csv",
        Path("/content/data/v5_probe_examples.csv"),
    ],
    "V5 probe examples",
)

CACHE_PATH = first_existing(
    [
        ROOT / "results" / "layer_curves" / "gpt2xl_hidden_states.pt",
        ROOT / "v5" / "results" / "layer_curves" / "gpt2xl_hidden_states.pt",
        Path("/content/results/layer_curves/gpt2xl_hidden_states.pt"),
    ],
    "V5 GPT-2 XL hidden-state cache",
)

ATLAS_PATH = first_existing(
    [
        ROOT / "artifacts" / "v5_decodability" / "decodability_atlas.csv",
        ROOT / "results" / "v5_decodability" / "decodability_atlas.csv",
        ROOT / "v5_decodability" / "decodability_atlas.csv",
        ROOT / "v5" / "artifacts" / "v5_decodability" / "decodability_atlas.csv",
        ROOT / "v5" / "results" / "v5_decodability" / "decodability_atlas.csv",
        Path("/content/results/v5_decodability/decodability_atlas.csv"),
    ],
    "V5 decodability atlas",
)

if DATA_PATH is None or CACHE_PATH is None or ATLAS_PATH is None:
    raise FileNotFoundError(
        "\nV6 needs the three V5 artifacts above. "
        "If they are in Google Drive, set ROOT to the SituatiONION project folder "
        "and rerun this cell."
    )

probe_examples = pd.read_csv(DATA_PATH)
atlas = pd.read_csv(ATLAS_PATH)

cache = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
tokenizer = cache["tokenizer"]
RUNS = cache["runs"]

print(f"\nLoaded {len(probe_examples):,} V5 sentence rows.")
print(f"Loaded {len(RUNS):,} cached V5 triples.")
print(f"Loaded {len(atlas):,} V5 atlas rows.")

EXPECTED_ATLAS_COLUMNS = {
    "task", "readout", "layer", "test_split", "control", "score"
}
missing = EXPECTED_ATLAS_COLUMNS - set(atlas.columns)
assert not missing, f"Atlas is missing columns: {sorted(missing)}"

expected_controls = {
    "linear", "majority", "lexical", "shuffled_labels", "random_projection"
}
assert expected_controls.issubset(set(atlas["control"].dropna().unique()))

assert set(ALL_TASKS).issubset(set(atlas["task"].dropna().unique()))
assert set(READOUTS).issubset(set(atlas.loc[atlas.control == "linear", "readout"].unique()))

print("\n✓ V5 compatibility checks passed.")
display(atlas.groupby("control").size().rename("n_rows"))


## 3. Reconstruct Conservative Advantage

For every:

$$
\text{task} 	imes \text{layer} 	imes \text{readout}
$$

extract the V5 scores:

- linear
- lexical
- random projection
- majority
- shuffled labels

Define the strongest nonchance control as:

$$
C^{*}=\max(C_{\text{lexical}}, C_{\text{random projection}})
$$

and conservative advantage as:

$$
a = BA_{\text{linear}}-C^{*}
$$

**Random projection is a compressed hidden-state control, not a chance baseline.**

Save the resulting table as `v6_accessibility_raw.parquet`.

In [ ]:
# Reconstruct conservative advantage from the ACTUAL long-form V5 atlas.
#
# Important V5 detail:
# - linear and random_projection are layer/readout-specific.
# - lexical TF-IDF is text-only, so V5 stores it at layer=-1/readout="lexical_tfidf".
# - majority is also global for the task/split and is NOT part of the
#   "strongest nonchance control" definition used in V5.7.

def build_v5_conservative_advantage(atlas: pd.DataFrame) -> pd.DataFrame:
    linear = (
        atlas.query("control == 'linear'")
        [["task", "readout", "layer", "test_split", "score"]]
        .rename(columns={"score": "linear_score"})
        .copy()
    )

    rp = (
        atlas.query("control == 'random_projection'")
        [["task", "readout", "layer", "test_split", "score"]]
        .rename(columns={"score": "random_projection_score"})
        .copy()
    )

    shuffled = (
        atlas.query("control == 'shuffled_labels'")
        [["task", "readout", "layer", "test_split", "score"]]
        .rename(columns={"score": "shuffled_label_score"})
        .copy()
    )

    # Lexical control is task × test_split only in V5.
    lexical = (
        atlas.query("control == 'lexical'")
        [["task", "test_split", "score"]]
        .drop_duplicates(["task", "test_split"])
        .rename(columns={"score": "lexical_score"})
        .copy()
    )

    majority = (
        atlas.query("control == 'majority'")
        [["task", "test_split", "score"]]
        .drop_duplicates(["task", "test_split"])
        .rename(columns={"score": "majority_score"})
        .copy()
    )

    keys = ["task", "readout", "layer", "test_split"]

    out = (
        linear
        .merge(rp, on=keys, how="left", validate="one_to_one")
        .merge(shuffled, on=keys, how="left", validate="one_to_one")
        .merge(lexical, on=["task", "test_split"], how="left", validate="many_to_one")
        .merge(majority, on=["task", "test_split"], how="left", validate="many_to_one")
    )

    out["strongest_nonchance_control"] = out[
        ["lexical_score", "random_projection_score"]
    ].max(axis=1, skipna=True)

    out["strongest_control_name"] = np.where(
        out["lexical_score"].fillna(-np.inf)
        >= out["random_projection_score"].fillna(-np.inf),
        "lexical",
        "random_projection",
    )

    out["conservative_advantage"] = (
        out["linear_score"] - out["strongest_nonchance_control"]
    )

    return out.sort_values(keys).reset_index(drop=True)

accessibility_raw = build_v5_conservative_advantage(atlas)

ACCESSIBILITY_PATH = RESULTS_DIR / "v6_accessibility_raw.parquet"
accessibility_raw.to_parquet(ACCESSIBILITY_PATH, index=False)

print(f"✓ Saved {len(accessibility_raw):,} V5-compatible accessibility rows")
print("  ", ACCESSIBILITY_PATH)

display(
    accessibility_raw.query("task in @ALL_TASKS")
    .head(10)
)


## V5 Hidden-State Compatibility Layer

V5 did **not** save a single `v5_hidden_states` matrix. It saved a GPT-2 XL cache with:

```text
RUNS[triple_id][variant]["hidden_states"][layer + 1]
```

The `+1` is important because Hugging Face's `hidden_states[0]` is the embedding output and V5 transformer layer 0 uses `hidden_states[1]`.

V6 reconstructs the same six readouts using the exact V5 logic below.

Also note that `agent_recipient` concatenates two token states and therefore has dimensionality **3200**, while the other readouts are **1600-dimensional**.


In [ ]:
# Exact V5 readout extraction logic, carried forward for V6.

EVENT = {
    "agent_recipient": ("gave", "gave", "gave"),
    "cause": ("called", "called", "called"),
    "temporal": ("cleaned", "cleaned", "cleaned"),
    "polarity": ("repair", "mend", "fix"),
    "event_state": ("carried", "transported", "dropped"),
}

CHANGED = {
    "temporal": ("after", "once", "before"),
    "polarity": ("repair", "mend", "not"),
    "event_state": ("carried", "transported", "dropped"),
}


def token_position(text, surface):
    matches = list(
        re.finditer(
            r"(?<![a-zA-Z])" + re.escape(str(surface)) + r"(?![a-zA-Z])",
            text,
            re.I,
        )
    )

    if not matches:
        return None

    return (
        len(
            tokenizer.encode(
                text[:matches[-1].end()],
                add_special_tokens=False,
            )
        )
        - 1
    )


def readout_vector(row, layer, readout):
    run = RUNS[row.triple_id][row.variant]
    hidden = run["hidden_states"][layer + 1]

    index = ("base", "paraphrase", "counterfactual").index(row.variant)

    if readout == "mean_pool":
        return hidden.mean(0).numpy()

    if readout == "max_pool":
        return hidden.max(0).values.numpy()

    if readout == "final_token":
        return hidden[-1].numpy()

    if readout == "event_token":
        pos = token_position(run["text"], EVENT[row.manipulation][index])
        return None if pos is None else hidden[pos].numpy()

    if readout == "changed_token":
        if (
            row.manipulation == "agent_recipient"
            and row.variant == "counterfactual"
        ):
            surface = row.semantic_recipient
        elif row.manipulation in {"agent_recipient", "cause"}:
            surface = row.semantic_agent
        else:
            surface = CHANGED[row.manipulation][index]

        pos = token_position(run["text"], surface)
        return None if pos is None else hidden[pos].numpy()

    if readout == "agent_recipient":
        agent_pos = token_position(run["text"], row.semantic_agent)
        recipient_pos = token_position(run["text"], row.semantic_recipient)

        if agent_pos is None or recipient_pos is None:
            return None

        return torch.cat(
            (hidden[agent_pos], hidden[recipient_pos])
        ).numpy()

    raise ValueError(f"Unknown readout: {readout}")


def build_representation_cache(readouts=READOUTS, layers=LAYERS):
    """
    Rebuild V5's task-independent readout matrices from the saved RUNS cache.

    Output:
        cache[(readout, layer)]["X"]      -> float32 matrix
        cache[(readout, layer)]["valid"]  -> valid-example mask
    """
    representation_cache = {}
    n_examples = len(probe_examples)

    for readout in readouts:
        for layer in layers:
            vectors = []
            valid = []

            for row in probe_examples.itertuples(index=False):
                vec = readout_vector(row, layer, readout)
                ok = vec is not None
                valid.append(ok)
                vectors.append(
                    None if vec is None else np.asarray(vec, dtype=np.float32)
                )

            first_valid = next((v for v in vectors if v is not None), None)

            if first_valid is None:
                representation_cache[(readout, layer)] = {
                    "X": None,
                    "valid": np.asarray(valid, dtype=bool),
                }
                continue

            X = np.zeros(
                (n_examples, first_valid.shape[0]),
                dtype=np.float32,
            )

            for row_i, vec in enumerate(vectors):
                if vec is not None:
                    X[row_i] = vec

            representation_cache[(readout, layer)] = {
                "X": X,
                "valid": np.asarray(valid, dtype=bool),
            }

    return representation_cache


index_to_position = {
    idx: pos
    for pos, idx in enumerate(probe_examples.index)
}


def get_representation_subset(indices, labels, readout, layer):
    cached = representation_cache[(readout, layer)]
    X = cached["X"]
    valid = cached["valid"]

    if X is None:
        return None, None

    positions = np.asarray(
        [index_to_position[idx] for idx in indices],
        dtype=int,
    )
    labels = np.asarray(labels)
    keep = valid[positions]

    if keep.sum() == 0:
        return None, None

    return X[positions[keep]], labels[keep]


# Building all 6×48 matrices can take time and RAM.
# Run once, then keep this notebook/runtime alive or add disk caching later.
representation_cache = build_representation_cache()

print(f"✓ Built {len(representation_cache)} V5-compatible representation matrices.")
for r in READOUTS:
    X0 = representation_cache[(r, 0)]["X"]
    print(r, None if X0 is None else X0.shape)


## Shared Statistical Utilities

These helpers are defined once and reused inside each experiment. Statistical tests are interpreted **inside the section that motivates them**, not postponed to a separate omnibus statistics section.

**Rule:** use example-level resampling whenever the estimator permits it; do not treat three relational tasks and two simple tasks as a large independent sample.

## 36. Bootstrap Uncertainty

Bootstrap at the **example level where valid** for:

- migration
- geometry
- generalization
- entity invariance
- coupling statistics

Store:

- estimate
- CI low
- CI high

In [ ]:
N_BOOTSTRAP = 2000

def bootstrap_ci(values, statistic=np.mean, n_boot=N_BOOTSTRAP, seed=SEED, alpha=0.05):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(statistic(sample))

    boots = np.asarray(boots)
    estimate = statistic(values)
    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return float(estimate), float(lo), float(hi)

# IMPORTANT:
# This generic helper is NOT automatically valid for every V6 statistic.
# Use example-level resampling matched to the exact estimator whenever possible.

## 37. Permutation / Null Testing

Build hypothesis-specific nulls.

Possible examples:

- shuffle relation labels
- shuffle layer ordering
- shuffle entity-role assignments

Do not use one generic permutation test for every scientific claim.

In [ ]:
N_PERMUTATIONS = 2000

# TODO — Implement separate null generators per hypothesis.
# Each test should record:
# observed_statistic
# null_mean
# null_sd
# p_value
# seed
# n_permutations

# PART II — V6.1 Relational Representational Migration

### Preregistered question

> **Does control-adjusted relational accessibility redistribute across readouts with depth more than simple-variable accessibility does?**

### Primary quantity

For task/relation $R$ and layer $l$:

$$
A_{R,l}=[a_1,\ldots,a_6]
$$

where each $a_r$ is the V5-style conservative advantage for one readout. Negative advantages are clipped to zero **only for constructing the migration distribution**:

$$
P_{R,l}(r)=
\frac{\max(a_r,0)}
{\sum_j \max(a_j,0)}
$$

if at least one component is positive. All-zero layers are marked undefined rather than forced to a uniform distribution.

Adjacent migration is:

$$
\boxed{
M_R(l)=JS(P_{R,l},P_{R,l+1})
}
$$

### Primary hypothesis

$$
\overline{M}_{\text{relational}}
>
\overline{M}_{\text{simple}}
$$

### Primary null / uncertainty / figure

- **Null:** relational and simple tasks do not differ systematically in migration.
- **Uncertainty:** example-level bootstrap where the full estimator can be recomputed; task-level summaries are descriptive because $3$ vs. $2$ task families is small.
- **Required figure:** migration-by-layer curves plus a continuous readout-accessibility heatmap.

### Secondary metrics

Accessibility entropy and one preregistered persistence measure. L1/cosine migration are robustness checks only; they are not alternative primaries selected after seeing results.

**Literature connection:** Chang et al. (2025) motivates depth-wise information-flow analysis. V6 extends the question from **how much information is available** to **where control-validated relational information is accessible across readouts**.

## Build Accessibility Profiles

For relation $R$ and layer $l$:

$$
A_{R,l} = [a_1,a_2,\ldots,a_6]
$$

where each $a_r$ is conservative advantage for one readout.

Effective shape:

$$
\text{task} 	imes \text{layer} 	imes \text{readout}
$$

For the five tasks and six readouts, this is approximately:

$$
5 	imes 48 	imes 6
$$

In [ ]:
# Accessibility profiles must preserve V5's held-out split.
# A task can have test_template, test_entity, and test_both results.
#
# Therefore the true object is:
#   task × test_split × layer × readout
#
# We do NOT average the three generalization conditions together.

def build_accessibility_matrix(
    accessibility_raw: pd.DataFrame,
    task: str,
    test_split: str,
) -> pd.DataFrame:

    sub = accessibility_raw.query(
        "task == @task and test_split == @test_split"
    )

    matrix = (
        sub.pivot_table(
            index="layer",
            columns="readout",
            values="conservative_advantage",
            aggfunc="mean",
        )
        .reindex(index=range(N_LAYERS), columns=READOUTS)
    )

    return matrix

available_task_splits = (
    accessibility_raw.query("task in @ALL_TASKS")
    [["task", "test_split"]]
    .drop_duplicates()
    .sort_values(["task", "test_split"])
)

accessibility_matrices = {
    (row.task, row.test_split):
        build_accessibility_matrix(accessibility_raw, row.task, row.test_split)
    for row in available_task_splits.itertuples(index=False)
}

print(f"Built {len(accessibility_matrices)} task × split accessibility matrices.")
display(available_task_splits)


## Normalize Positive Accessibility

For migration only, clip negative conservative advantage to zero:

$$
A^+_{R,l} = \max(A_{R,l},0)
$$

Then normalize positive values:

$$
P_{R,l} = \frac{A^+_{R,l}}{\sum_r A^+_{R,l,r}}
$$

If all six values are zero, leave the distribution undefined (`NaN`).

**Do not replace an all-zero row with a uniform distribution.**

In [ ]:
def positive_accessibility_distribution(values):
    values = np.asarray(values, dtype=float)
    positive = np.maximum(values, 0.0)
    total = np.nansum(positive)

    if not np.isfinite(total) or total <= 0:
        return np.full_like(positive, np.nan, dtype=float)

    return positive / total

# Example:
# P = positive_accessibility_distribution(accessibility_matrices["temporal_relation"].iloc[10].values)

## Compute Jensen–Shannon Migration

For each adjacent layer transition:

$$
M_R(l)=JS(P_{R,l},P_{R,l+1})
$$

Compute transitions:

$$
0
ightarrow1,\ 1
ightarrow2,\ \ldots,\ 46
ightarrow47
$$

Output columns:

`task | layer_from | layer_to | JS_migration`

In [ ]:
def js_migration(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    if np.isnan(p).any() or np.isnan(q).any():
        return np.nan

    # scipy.spatial.distance.jensenshannon returns sqrt(JS divergence).
    # Squaring gives the Jensen-Shannon divergence used as M_R(l).
    return float(jensenshannon(p, q, base=2.0) ** 2)


def compute_migration_table(accessibility_matrices):
    rows = []

    for (task, test_split), mat in accessibility_matrices.items():
        probs = np.vstack([
            positive_accessibility_distribution(mat.loc[layer].values)
            for layer in range(N_LAYERS)
        ])

        for layer in range(N_LAYERS - 1):
            rows.append({
                "task": task,
                "test_split": test_split,
                "layer_from": layer,
                "layer_to": layer + 1,
                "JS_migration": js_migration(
                    probs[layer],
                    probs[layer + 1]
                ),
            })

    return pd.DataFrame(rows)


migration_df = compute_migration_table(accessibility_matrices)
MIGRATION_PATH = RESULTS_DIR / "v6_migration.parquet"
migration_df.to_parquet(MIGRATION_PATH, index=False)

print(f"✓ Saved migration table: {MIGRATION_PATH}")
display(migration_df.head())


## Compute Accessibility Entropy

For each valid accessibility distribution:

$$
H_R(l)=-\sum_rP_r\log P_r
$$

Interpretation:

- **low entropy:** accessibility is concentrated in one or a few readouts
- **high entropy:** accessibility is distributed across readouts

In [ ]:
def accessibility_entropy(p):
    p = np.asarray(p, dtype=float)
    if np.isnan(p).any():
        return np.nan
    return float(entropy(p, base=2))

# TODO:
# Build task × layer entropy table and save to RESULTS_DIR.

## Compute Persistence

Measure how similar the accessibility distribution remains over longer depth intervals.

At minimum evaluate:

$$
\text{similarity}(P_l,P_{l+k})
$$

for several values of $k$, such as $k=1,2,4,8$.

Use one preregistered primary similarity metric and treat alternatives as robustness checks.

In [ ]:
# TODO — Choose and preregister one primary persistence metric.
# Example option: 1 - Jensen-Shannon divergence.

PERSISTENCE_LAGS = [1, 2, 4, 8]

def persistence_similarity(p, q):
    if np.isnan(p).any() or np.isnan(q).any():
        return np.nan
    return 1.0 - js_migration(p, q)

# TODO:
# Return columns:
# task, layer_from, layer_to, lag, persistence_similarity

## Migration Results and Figures

Generate:

- `migration_role_assignment.png`
- `migration_cause_holder.png`
- `migration_temporal.png`
- `migration_controls.png`
- `accessibility_heatmaps.png`
- `entropy_by_layer.png`

The most important visualization is the continuous accessibility heatmap:

- **x:** layer
- **y:** readout
- **value:** conservative advantage

In [ ]:
# TODO — Publication figure functions.
# Do not hard-code interpretations into the plot.

def plot_accessibility_heatmap(matrix: pd.DataFrame, title: str, save_path: Path):
    fig, ax = plt.subplots(figsize=(14, 4))
    im = ax.imshow(matrix.T.values, aspect="auto", origin="lower")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Readout")
    ax.set_yticks(range(len(matrix.columns)))
    ax.set_yticklabels(matrix.columns)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="Conservative advantage")
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

print("TODO: create migration/entropy/heatmap figures.")

# PART III — V6.2 Emergence of Relational Abstraction

### Preregistered question

> **Does the relational signal survive systematically novel combinations rather than familiar lexical/entity/template patterns?**

### Primary quantity

$$
\boxed{
G_R(l,k)
=
\text{conservative advantage for relation }R
\text{ at layer }l
\text{ under holdout level }k
}
$$

Holdout difficulty:

```text
K0  existing/baseline split
K1  unseen entities
K2  unseen entity pairs
K3  unseen predicates / verbs
K4  unseen templates
K5  unseen entity × verb combinations
K6  unseen entity × verb × template combinations
```

### Primary hypothesis

Relational conservative advantage remains positive under the hardest **valid, adequately powered** compositional holdouts.

### Primary null / uncertainty / figure

- **Null:** V5 relational advantage collapses once familiar lexical/entity/template combinations are removed.
- **Uncertainty:** bootstrap over held-out examples within each fixed split.
- **Required figure:** relation-specific **layer × holdout-difficulty abstraction maps**.

### Non-negotiable validity check

The split generator is part of the experiment. Leakage checks must pass **before any probe is fit**. If a holdout produces tiny or badly imbalanced sets, report it as unsupported rather than forcing the analysis.

**Literature connection:** Petty et al. (2024) motivates a depth–compositionality link; Wold et al. (2024) motivates controlled novel-combination splits. V6 asks the within-model question: **where across depth does systematic relational abstraction become recoverable?**

## Build Expanded Structured Stimuli

Each stimulus must contain explicit metadata:

- `entity_1`
- `entity_2`
- `verb`
- `predicate`
- `template`
- `syntax_family`
- `relation_type`
- `relation_label`

Use substantially more examples than the small V5 headline test sets.

In [ ]:
REQUIRED_STIMULUS_COLUMNS = [
    "example_id",
    "text",
    "entity_1",
    "entity_2",
    "verb",
    "predicate",
    "template",
    "syntax_family",
    "relation_type",
    "relation_label",
]

# TODO — Load or generate V6 stimuli into v6_stimuli.
# assert set(REQUIRED_STIMULUS_COLUMNS).issubset(v6_stimuli.columns)

## Construct Deterministic Holdouts

Use increasingly difficult systematic holdouts:

- **K0:** normal random split
- **K1:** unseen entities
- **K2:** unseen entity pairs
- **K3:** unseen predicates / verbs
- **K4:** unseen templates
- **K5:** unseen entity × verb combinations
- **K6:** unseen entity × verb × template combinations

Save all split assignments so they are exactly reproducible.

In [ ]:
HOLDOUT_LEVELS = {
    "K0": "random",
    "K1": "unseen_entities",
    "K2": "unseen_entity_pairs",
    "K3": "unseen_predicates_verbs",
    "K4": "unseen_templates",
    "K5": "unseen_entity_x_verb",
    "K6": "unseen_entity_x_verb_x_template",
}

# TODO — Implement deterministic split constructors for each holdout.
# Return a dataframe with:
# example_id, holdout, split

## Leakage Audit — Must Pass Before Probing

Programmatically verify each holdout.

Examples:

```python
assert no_test_entity_in_training_for_K1
assert no_test_pair_in_training_for_K2
assert no_test_template_in_training_for_K4
```

If leakage is detected, **halt the notebook** rather than continuing with invalid results.

In [ ]:
def fail_if_overlap(train_values, test_values, label):
    overlap = set(train_values) & set(test_values)
    if overlap:
        raise RuntimeError(
            f"Leakage detected for {label}: {len(overlap)} overlapping values."
        )

# TODO — Add one leakage audit per holdout level.
# Print a compact PASS/FAIL report.

## Run Layer × Readout × Holdout Probes

For every:

$$
\text{relation} 	imes \text{layer} 	imes \text{readout} 	imes \text{holdout}
$$

calculate:

- linear balanced accuracy
- majority baseline
- lexical control
- random projection control
- shuffled-label control
- conservative advantage

This produces:

$$
G_R(l,k)
$$

In [ ]:
# TODO — Reuse the V5 probing logic exactly where possible.
#
# Output columns should include:
# task, layer, readout, holdout,
# n_train, n_test,
# linear_score,
# majority_score,
# lexical_score,
# random_projection_score,
# shuffled_label_score,
# strongest_nonchance_control,
# conservative_advantage

print("TODO: run depth-resolved compositional holdout probes.")

## Abstraction Results and Maps

For each relation, create a heatmap with:

- **rows:** holdout difficulty $k$
- **columns:** layer $l$
- **value:** conservative advantage

These maps show **when** increasingly systematic relational generalization becomes recoverable across depth.

In [ ]:
# TODO — Produce one abstraction heatmap per relation.
# Recommended source table:
# task × layer × holdout × conservative_advantage

# PART IV — V6.3 Relational Geometric Dynamics

### Preregistered question

> **Does the geometry supporting each validated relation remain stable, or does it systematically transform across depth?**

### Primary quantity

For relation $R$, estimate a task-relevant subspace $S_R(l)$ at each layer using a **fixed, preregistered readout** and a frozen subspace-estimation procedure.

Adjacent geometric change:

$$
\boxed{
D_R(l)=d_G\!\left(S_R(l),S_R(l+1)\right)
}
$$

where $d_G$ is the preregistered Grassmann / principal-angle distance.

### Primary hypothesis

No directional result is assumed. The scientifically meaningful alternatives are:

1. **stable relation geometry** across depth;
2. **localized transformation** over a subset of layers;
3. **continuous transformation** across depth.

### Primary null / uncertainty / figure

- **Null:** observed adjacent movement is no more structured than an appropriate label/subspace null.
- **Uncertainty:** bootstrap examples and/or permutation tests matched to the fitted subspace estimator.
- **Required figure:** adjacent $D_R(l)$ curves and a full 48 × 48 cross-layer similarity matrix for each relation.

### Secondary analyses

- CKA / SVCCA as robustness only.
- Cross-relation geometry (`role↔cause`, `role↔time`, `cause↔time`) is secondary to **same-relation cross-depth trajectory**.
- Simple variables (`event_state`, `polarity`) are reference geometries.

**Literature connection:** Chanin et al. (2024) motivates linear relational structure; Sakata et al. (2026) motivates cross-layer relational geometry; V6 asks how **control-validated situation-relation geometry itself changes across depth**.

## Build Matched Geometry Datasets

For each:

$$
\text{relation} 	imes \text{layer} 	imes \text{chosen fixed readout}
$$

construct:

- $X$: hidden states
- $y$: relation labels

For the primary cross-layer analysis, use a **fixed semantically coherent readout**.

Do not compare `changed_token @ L16` directly against `max_pool @ L38` and treat the difference as pure layer geometry because that confounds **layer** and **readout**.

In [ ]:
# Primary geometry readout.
# Keep fixed across depth so layer change is not confounded with readout change.
PRIMARY_GEOMETRY_READOUT = "changed_token"


def get_geometry_dataset(
    task,
    layer,
    readout=PRIMARY_GEOMETRY_READOUT,
    split="dev",
):
    """
    Return example-level V5 representations for one task/layer/readout.

    Geometry is estimated on V5's DEV data by default so held-out test
    examples remain available for generalization evaluation.
    """
    if task not in TASKS:
        raise KeyError(f"Unknown V5 task: {task}")

    label_col, eligible_col = TASKS[task]

    data = probe_examples[
        probe_examples[eligible_col]
        & (probe_examples["split"] == split)
    ].copy()

    X, y = get_representation_subset(
        data.index.to_numpy(),
        data[label_col].to_numpy(),
        readout,
        layer,
    )

    if X is None:
        return None, None, data.iloc[0:0].copy()

    # Reconstruct the valid-row metadata mask exactly.
    cached = representation_cache[(readout, layer)]
    positions = np.asarray(
        [index_to_position[idx] for idx in data.index],
        dtype=int,
    )
    keep = cached["valid"][positions]
    meta = data.loc[data.index[keep]].copy()

    assert len(X) == len(y) == len(meta)

    return X, y, meta


X_demo, y_demo, meta_demo = get_geometry_dataset(
    "cause_holder_role",
    layer=16,
)

print("Geometry demo:", X_demo.shape, y_demo.shape, meta_demo.shape)


## Estimate the Preregistered Relational Subspace

For every relation and layer, estimate:

$$
S_R(l)
$$

using **one preregistered primary method**.

A simple starting point is a linear discriminative subspace such as LDA or a low-rank supervised projection.

Save the learned basis for every layer.

In [ ]:
# TODO — Primary relational subspace estimator.
# For binary tasks, note that ordinary LDA may produce only one discriminant direction.
# If you want k>1 dimensions, use a deliberately defined low-rank supervised method.

GEOMETRY_SUBSPACE_DIM = 1

def fit_relational_subspace(X, y, n_components=GEOMETRY_SUBSPACE_DIM):
    lda = LinearDiscriminantAnalysis(n_components=min(n_components, len(np.unique(y)) - 1))
    lda.fit(X, y)

    # scalings_ columns span the discriminative directions in feature space.
    basis = lda.scalings_[:, :lda._max_components]

    # Orthonormalize basis for principal-angle calculations.
    q, _ = np.linalg.qr(basis)
    return q

# relation_subspaces = {task: {} for task in RELATIONAL_TASKS}
# for task in RELATIONAL_TASKS:
#     for layer in range(N_LAYERS):
#         X, y, meta = get_geometry_dataset(...)
#         relation_subspaces[task][layer] = fit_relational_subspace(X, y)

## Compute Adjacent Geometric Movement

Compute principal angles:

$$
	heta_1,\ldots,	heta_k
$$

between:

$$
S_R(l)\quad\text{and}\quad S_R(l+1)
$$

Define the primary geometric movement statistic as a preregistered Grassmann/principal-angle distance:

$$
D_R(l)=d_G(S_R(l),S_R(l+1))
$$

Save:

`task | layer_from | layer_to | geometric_distance`

In [ ]:
def grassmann_distance(U, V):
    angles = subspace_angles(U, V)
    return float(np.linalg.norm(angles))

def compute_adjacent_geometry(relation_subspaces):
    rows = []
    for task, layer_map in relation_subspaces.items():
        for layer in range(N_LAYERS - 1):
            U = layer_map[layer]
            V = layer_map[layer + 1]
            rows.append({
                "task": task,
                "layer_from": layer,
                "layer_to": layer + 1,
                "geometric_distance": grassmann_distance(U, V),
            })
    return pd.DataFrame(rows)

# geometry_df = compute_adjacent_geometry(relation_subspaces)
# geometry_df.to_parquet(RESULTS_DIR / "v6_geometry_adjacent.parquet", index=False)

## Compute Full Cross-Layer Geometry

For each relation, compare every layer against every other layer to obtain a:

$$
48	imes48
$$

similarity or distance matrix.

This can reveal:

- gradual rotation
- stable blocks
- abrupt changes
- convergence
- late reorganization

In [ ]:
def cross_layer_geometry_matrix(layer_map):
    M = np.full((N_LAYERS, N_LAYERS), np.nan, dtype=float)
    for i in range(N_LAYERS):
        for j in range(N_LAYERS):
            M[i, j] = grassmann_distance(layer_map[i], layer_map[j])
    return M

# cross_layer_geometry = {
#     task: cross_layer_geometry_matrix(relation_subspaces[task])
#     for task in RELATIONAL_TASKS
# }

## Geometry Robustness Checks

After the primary geometric metric is fixed and working, add secondary checks such as:

- CKA
- CCA / SVCCA
- cosine similarity between probe directions

These are **robustness analyses**, not metrics to shop between after seeing results.

In [ ]:
# TODO — Add robustness implementations only after the primary geometry pipeline is frozen.

def linear_cka(X, Y):
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    numerator = np.linalg.norm(X.T @ Y, ord="fro") ** 2
    denominator = (
        np.linalg.norm(X.T @ X, ord="fro")
        * np.linalg.norm(Y.T @ Y, ord="fro")
    )
    return float(numerator / denominator) if denominator > 0 else np.nan

## Secondary Cross-Relation Geometry

At each layer compare:

$$
S_{\text{role}}(l)\leftrightarrow S_{\text{cause}}(l)
$$

$$
S_{\text{role}}(l)\leftrightarrow S_{\text{temporal}}(l)
$$

$$
S_{\text{cause}}(l)\leftrightarrow S_{\text{temporal}}(l)
$$

This asks whether different relation families remain distinct or become more geometrically similar with depth.

In [ ]:
RELATION_PAIRS = [
    ("role_assignment", "cause_holder_role"),
    ("role_assignment", "temporal_relation"),
    ("cause_holder_role", "temporal_relation"),
]

# TODO — Compute one geometric distance per pair × layer.
# Output columns:
# relation_a, relation_b, layer, geometric_distance

# PART V — V6.4 Migration–Geometry Coupling

### Preregistered question

> **When relational accessibility redistributes across readouts, does the underlying relational geometry also transform?**

For each adjacent transition:

$$
(M_R(l),D_R(l)).
$$

### Primary analysis

Estimate the association between $M_R(l)$ and $D_R(l)$ with Pearson and Spearman summaries, but designate **one** as primary before inspecting results.

### Primary null / uncertainty / figure

- **Null:** migration and geometric change are unrelated after respecting task/layer structure.
- **Null construction:** shuffle transition alignment within task or use another task-aware permutation that preserves marginal trajectories.
- **Uncertainty:** bootstrap transitions/examples in a way consistent with how $M$ and $D$ were estimated.
- **Required figure:** transition scatter with task identity plus layerwise paired trajectories.

### Interpretation grid

| Migration | Geometry | Interpretation |
|---|---|---|
| low | low | stable accessibility and geometry |
| high | low | readout redistribution of relatively stable geometry |
| low | high | internal geometric transformation without strong readout relocation |
| high | high | coupled representational reorganization |

Association is **not causation**.

## Merge Migration and Geometry

For every relation and adjacent layer transition, combine:

- migration $M_R(l)$
- geometric change $D_R(l)$
- accessibility entropy
- conservative advantage summaries

In [ ]:
# TODO — Merge migration_df, geometry_df, entropy table, and accessibility summaries.

# coupling_df = (
#     migration_df
#     .merge(geometry_df, on=["task", "layer_from", "layer_to"], how="inner")
#     .merge(...)
# )

print("TODO: build migration–geometry coupling table.")

## Test Migration–Geometry Coupling

Calculate:

- Pearson correlation
- Spearman correlation
- example-valid bootstrap confidence intervals

between:

$$
M_R(l)
$$

and:

$$
D_R(l)
$$

Run the analysis separately for each relational task and compare the pattern against `event_state` and `polarity`.

In [ ]:
def correlation_summary(df, x_col="JS_migration", y_col="geometric_distance"):
    clean = df[[x_col, y_col]].dropna()
    if len(clean) < 3:
        return {"n": len(clean), "pearson_r": np.nan, "pearson_p": np.nan,
                "spearman_r": np.nan, "spearman_p": np.nan}

    pr, pp = pearsonr(clean[x_col], clean[y_col])
    sr, sp = spearmanr(clean[x_col], clean[y_col])

    return {
        "n": len(clean),
        "pearson_r": pr,
        "pearson_p": pp,
        "spearman_r": sr,
        "spearman_p": sp,
    }

# TODO — Add preregistered bootstrap scheme.

## Coupling Results and Transition Types

Use the migration–geometry scatterplot to visualize four conceptual regimes:

| Migration | Geometry | Interpretation |
|---|---|---|
| low | low | stable |
| high | low | accessibility redistribution |
| low | high | geometric transformation |
| high | high | representational reorganization |

Do not turn these into hard scientific categories unless thresholds are specified **before** inspecting results.

In [ ]:
# TODO — Plot JS migration vs geometric distance.
# Prefer continuous values.
# If threshold quadrants are displayed, define thresholds in configuration first.

# PART VI — Relation Lens — Secondary Interpretability Analysis

### Question

> **Can hidden states be mapped into a common, interpretable relational space that generalizes across entities and remains comparable across depth?**

This is a **secondary** V6 analysis. It must not block the three primary experiments above.

### Required design constraint

Do **not** train 48 unrelated classifiers and call the result a trajectory. Choose one explicit cross-layer scheme before running:

- shared transform across pooled training layers, or
- anchor-layer transform applied across depth,

with the alternative used only as a prespecified robustness check.

### Primary metric

Held-out relation recovery in the common lens space, ideally under **entity-held-out evaluation**.

### Primary null / uncertainty / figure

- **Null:** the lens does not generalize to held-out entities better than matched controls.
- **Uncertainty:** bootstrap held-out entities/examples.
- **Required figure:** relation-lens recovery by layer.

### Entity invariance

For matched relational transformations:

$$
\Delta_R(e,l)
$$

compare direction similarity across entities. High agreement supports entity-invariant relational structure; low agreement supports entity-specific encoding.

**Literature connection:** Morand, Mothe & Piwowarski (2025) motivate an Entity Lens for internal entity representations. SituatiONION extends the idea toward a **depth-resolved Relation Lens**.

## Build Relation Lens Data

For every example collect:

- hidden state
- relation type
- relation label
- entity identity
- template
- verb
- layer
- readout

Example:

```text
Alice broke the vase.
relation = cause_holder
label = AGENT
entity = Alice
```

In [ ]:
# Relation Lens metadata mapped to V5.
#
# V5 already supplies:
#   triple_id, variant, text, semantic_agent, semantic_recipient,
#   manipulation, split, task labels, and eligibility flags.
#
# New V6 compositional stimuli should later add explicit verb/template fields.

RELATION_LENS_COLUMNS = [
    "triple_id",
    "variant",
    "text",
    "semantic_agent",
    "semantic_recipient",
    "manipulation",
    "split",
]

missing = [c for c in RELATION_LENS_COLUMNS if c not in probe_examples.columns]

if missing:
    print("V5 metadata columns not present:", missing)
else:
    print("✓ Core V5 Relation Lens metadata is available.")

display(probe_examples.head())


## Entity-Held-Out Split

Entity identity must be held out explicitly.

Example:

**Train entities:** Alice, Bob, Carol, David  
**Test entities:** Priya, Marcus, Sofia, Ethan

Test entities must never appear in training for the entity-invariance evaluation.

In [ ]:
# TODO — Replace with metadata-derived entity groups rather than hard-coded names.

def split_entities(metadata, entity_col="entity_identity", test_size=0.25, seed=SEED):
    entities = np.array(sorted(metadata[entity_col].dropna().unique()))
    train_entities, test_entities = train_test_split(
        entities,
        test_size=test_size,
        random_state=seed
    )
    return set(train_entities), set(test_entities)

# train_entities, test_entities = split_entities(relation_lens_meta)
# assert train_entities.isdisjoint(test_entities)

## Learn the Shared Relational Transformation

Learn:

$$
T_R:\mathbb{R}^{1600}
ightarrow\mathbb{R}^{k}
$$

with a low-dimensional relational space such as $k=8$ or $k=16$.

The goal is for examples with the **same relational state** to become similar even when they contain different entities.

In [ ]:
RELATION_LENS_DIM = 8

class LinearRelationLens:
    """
    Minimal scaffold for a low-dimensional linear relation lens.

    Replace/extend this once the exact training objective is chosen.
    """
    def __init__(self, dim=RELATION_LENS_DIM):
        self.dim = dim
        self.projection_ = None

    def fit(self, X, y):
        # TODO:
        # Replace with a preregistered supervised low-rank objective.
        # This placeholder uses LDA where possible.
        n_classes = len(np.unique(y))
        n_components = min(self.dim, n_classes - 1)
        lda = LinearDiscriminantAnalysis(n_components=n_components)
        Z = lda.fit_transform(X, y)

        self.projection_ = lda.scalings_[:, :n_components]
        self.classes_ = np.array(sorted(np.unique(y)))

        # Orthonormalized projection
        self.projection_, _ = np.linalg.qr(self.projection_)
        return self

    def transform(self, X):
        if self.projection_ is None:
            raise RuntimeError("Fit the lens first.")
        return X @ self.projection_

## Learn Relational Prototypes

For each relation label:

$$
\mu_y=\frac{1}{N_y}\sum_i z_i
$$

where:

$$
z_i=T_R(h_i)
$$

Classify unseen examples by nearest prototype.

Examples:

- `prototype_agent`
- `prototype_recipient`
- `prototype_before`
- `prototype_after`

In [ ]:
def compute_prototypes(Z, y):
    prototypes = {}
    for label in np.unique(y):
        prototypes[label] = Z[y == label].mean(axis=0)
    return prototypes

def nearest_prototype_predict(Z, prototypes):
    labels = list(prototypes.keys())
    P = np.vstack([prototypes[label] for label in labels])

    preds = []
    for z in Z:
        d = np.linalg.norm(P - z[None, :], axis=1)
        preds.append(labels[int(np.argmin(d))])
    return np.asarray(preds)

## Apply One Consistent Lens Across Depth

Do not train 48 unrelated lenses and then interpret their outputs as one trajectory.

Use a **defined shared training scheme** so that cross-layer comparisons are meaningful.

Then obtain:

$$
L_R(l)
$$

where $L_R(l)$ is Relation Lens recovery at layer $l$.

In [ ]:
# TODO — Choose ONE explicit cross-layer lens scheme before running.
#
# Candidate schemes:
# A. Train one lens on a preregistered anchor layer and apply it to all layers.
# B. Learn one shared projection jointly from all training layers.
# C. Learn aligned layer-specific projections with a common prototype space.
#
# Do NOT silently switch schemes after seeing results.

RELATION_LENS_SCHEME = "anchor_layer"
RELATION_LENS_ANCHOR_LAYER = 24  # TODO: preregister/justify

print("Relation Lens scheme:", RELATION_LENS_SCHEME)

## Test Entity Invariance

For entity $e$, relation $R$, and layer $l$, compare a relation-conditioned transformation:

$$
\Delta_R(e,l)
$$

across entities.

Plain-language question:

> Does putting Alice into role X alter her representation in a similar direction to putting Bob into role X?

High cross-entity agreement supports **entity-independent relational structure**.

Low agreement supports more **entity-specific encoding**.

In [ ]:
# TODO — Define delta construction using matched sentence pairs.
#
# Example:
# delta(entity, layer) = h(entity in role A, layer) - h(entity in role B, layer)
#
# Then calculate pairwise cosine similarity between deltas for different entities.

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

## Relation Lens Results and Figures

Generate:

- `relation_lens_accuracy_by_layer.png`
- `entity_invariance_by_layer.png`
- `prototype_separation_by_layer.png`
- `relation_lens_vs_v5_probe.png`

In [ ]:
# TODO — Create Relation Lens visualizations from saved result tables.

# PART VII — Integrated V6 Analysis

### Central integration question

> **Where does relational abstraction emerge relative to migration and geometric transformation?**

Merge the section-level outputs into one master atlas keyed by task/relation, split/holdout, layer, and readout where applicable.

### Primary preregistered relationships

$$
M_R(l)\leftrightarrow G_R(l,k)
$$

$$
D_R(l)\leftrightarrow G_R(l,k)
$$

$$
M_R(l)\leftrightarrow D_R(l)
$$

Relation Lens and entity-invariance metrics are secondary additions to the master table.

### Lagged analysis

Lagged comparisons such as

$$
D_R(l)\leftrightarrow G_R(l+\Delta,k)
$$

are **exploratory temporal-order associations**, not causal evidence.

### Required figure

One integrated panel showing, for each relational task:

1. control-adjusted accessibility,
2. migration,
3. geometric movement,
4. abstraction under the hardest valid holdout.

This integrated figure should carry the main V6 story.

## Build the Master V6 Atlas

Create one master dataframe containing:

- relation
- layer
- readout
- conservative advantage
- migration
- entropy
- geometric change
- Relation Lens score
- entity invariance
- K0 generalization
- …
- K6 generalization

Save as:

`v6_master_atlas.parquet`

In [ ]:
# TODO — Merge all V6 result tables with explicit keys.
#
# Suggested keys:
# task, layer, readout where readout-specific
# task, layer_from, layer_to for transition metrics
#
# Avoid accidental many-to-many merges.

# master_atlas.to_parquet(RESULTS_DIR / "v6_master_atlas.parquet", index=False)

print("TODO: build and save V6 master atlas.")

## Test Preregistered Cross-Analysis Relationships

Primary integrations:

$$
M\leftrightarrow D
$$

Migration vs geometry.

$$
D\leftrightarrow G
$$

Geometry vs abstraction.

$$
M\leftrightarrow G
$$

Migration vs abstraction.

$$
L\leftrightarrow G
$$

Relation Lens recovery vs abstraction.

$$
E\leftrightarrow G
$$

Entity invariance vs abstraction.

In [ ]:
# TODO — Define preregistered correlation / regression analyses.
# Keep task-wise analyses separate before any pooled analysis.
# Report effect sizes and uncertainty, not only p-values.

## Exploratory Lagged Associations

Explore whether geometric change tends to occur **before, during, or after** abstraction emerges.

Examples:

$$
D(l)\leftrightarrow G(l+1)
$$

$$
D(l)\leftrightarrow G(l+2)
$$

This is a **temporally ordered association**, not a causal effect.

Do not write:

> geometry causes abstraction

Prefer:

> geometric change precedes or is associated with later abstraction

In [ ]:
LAG_VALUES = [0, 1, 2, 4]

# TODO — Compute preregistered lagged associations.
# Be explicit about boundary handling and multiple comparisons.

# PART VIII — Final Outputs

The final section should do only three things:

1. generate publication-quality figures and tables from already-saved result files;
2. print compact factual summaries with uncertainty;
3. state conclusions, null results, limitations, and the V7 handoff.

Do not introduce new metrics here.

## Compact Result Summaries

Automatically identify:

- largest migration transition
- largest geometric transition
- peak abstraction layer
- first layer surviving K6
- peak Relation Lens layer
- peak entity-invariance layer
- strongest role/cause similarity
- strongest temporal divergence

In [ ]:
def safe_idxmax(df, value_col):
    clean = df.dropna(subset=[value_col])
    if clean.empty:
        return None
    return clean.loc[clean[value_col].idxmax()].to_dict()

summary = {}

# TODO examples:
# summary["temporal_peak_migration"] = safe_idxmax(
#     migration_df.query("task == 'temporal_relation'"),
#     "JS_migration"
# )
#
# summary["cause_first_K6_layer"] = ...

summary

## Automatic Figure Captions

Generate factual captions from figure metadata.

Example:

> **Figure 7.** Control-adjusted temporal-relation accessibility across 48 GPT-2 XL layers. Darker cells indicate greater conservative advantage relative to the strongest nonchance control.

Captions should describe what is plotted, not declare the theoretical conclusion.

In [ ]:
# TODO — Store figure metadata and generate factual captions programmatically.

figure_captions = {}

## Final Figures

Create one function that regenerates every final figure from cached result tables:

```python
make_all_figures(master_atlas)
```

In [ ]:
def make_all_figures(master_atlas):
    """
    TODO:
    Call all final plotting functions from one place.
    Every figure should save to FIGURE_DIR.
    """
    raise NotImplementedError

# make_all_figures(master_atlas)

## Final Tables

Automatically create:

1. V5 → V6 baseline
2. Migration statistics
3. Geometry statistics
4. Relation Lens results
5. Compositional generalization
6. Integrated relationships

In [ ]:
# TODO — Build publication-ready result dataframes and export CSV/LaTeX/Markdown as needed.

FINAL_TABLE_NAMES = [
    "table1_v5_to_v6_baseline",
    "table2_migration",
    "table3_geometry",
    "table4_relation_lens",
    "table5_compositional_generalization",
    "table6_integrated_relationships",
]

## Final Conclusions and V7 Handoff

Have Python print the final numerical result summary.

Write the scientific interpretation manually after examining:

- effect sizes
- uncertainty
- robustness checks
- control comparisons
- null tests

Do **not** automate conclusions using rules like:

```python
if p < .05:
    print("The model forms abstract relations!")
```

The notebook should compute evidence. The interpretation remains a scientific judgment.

In [ ]:
# TODO — Print compact numeric summary only.
# Example:
#
# print(json.dumps(summary, indent=2, default=str))
#
# Then add a NEW MARKDOWN CELL manually containing your scientific interpretation.

# References

### Core papers used to design V6

**Chanin, D., Hunter, A., & Camburu, O.-M. (2024).**  
_Identifying Linear Relational Concepts in Large Language Models._  
NAACL 2024, pp. 1524–1535.  
DOI: `10.18653/v1/2024.naacl-long.85`  
https://aclanthology.org/2024.naacl-long.85/

**Morand, V., Mothe, J., & Piwowarski, B. (2025).**  
_On the Representations of Entities in Auto-regressive Large Language Models._  
BlackboxNLP 2025, pp. 433–451.  
DOI: `10.18653/v1/2025.blackboxnlp-1.25`  
https://aclanthology.org/2025.blackboxnlp-1.25/

**Sakata, M., Heinzerling, B., Ito, T., Yokoi, S., & Inui, K. (2026).**  
_Linear Representations of Hierarchical Concepts in Language Models._  
arXiv:2604.07886.  
https://arxiv.org/abs/2604.07886

**Chang, R., Deng, C., & Chen, H. (2025).**  
_The Generalization Ridge: Information Flow in Natural Language Generation._  
arXiv:2507.05387.  
https://arxiv.org/abs/2507.05387

**Petty, J., van Steenkiste, S., Dasgupta, I., Sha, F., Garrette, D., & Linzen, T. (2024).**  
_The Impact of Depth on Compositional Generalization in Transformer Language Models._  
NAACL 2024, pp. 7239–7252.  
DOI: `10.18653/v1/2024.naacl-long.402`  
https://aclanthology.org/2024.naacl-long.402/

**Wold, S., Simon, É., Charpentier, L., Kostylev, E., Velldal, E., & Øvrelid, L. (2024).**  
_Compositional Generalization with Grounded Language Models._  
Findings of ACL 2024.  
https://aclanthology.org/2024.findings-acl.205/

---

### How these papers map onto SituatiONION V6

| Prior work | What V6 borrows | SituatiONION extension |
|---|---|---|
| Chang et al. (2025) | depth-wise information-flow perspective | relation-specific accessibility **distribution and migration across readouts** |
| Chanin et al. (2024) | interpretable linear relational structure | relation **trajectory** across layers |
| Sakata et al. (2026) | cross-layer relational geometry and unseen-data tests | non-hierarchical **situation relations**, migration–geometry–abstraction coupling |
| Morand et al. (2025) | Entity Lens / entity reconstruction perspective | **Relation Lens** and entity-invariant relation-conditioned structure |
| Petty et al. (2024) | depth and compositional generalization | within-one-model **layer-wise emergence of relational abstraction** |
| Wold et al. (2024) | controlled novel-combination evaluation | K0–K6 systematic relational holdouts |

These papers motivate the experimental design; they do not predetermine the expected V6 result.
